In [1]:
%%capture
!pip install fitz
!pip install transformers
!pip install PyMuPDF
!pip install torch
!pip install spacy

##**FaceBook Bart - Large - CNN**

BART is a transformer encoder-encoder (seq2seq) model with a bidirectional (BERT-like) encoder and an autoregressive (GPT-like) decoder. BART is pre-trained by corrupting text with an arbitrary noising function, and learning a model to reconstruct the original text.

BART is particularly effective when fine-tuned for text generation (e.g. summarization, translation) but also works well for comprehension tasks (e.g. text classification, question answering). This particular checkpoint has been fine-tuned on CNN Daily Mail, a large collection of text-summary pairs.

In [2]:
import os
import fitz
import torch
from transformers import AutoTokenizer, AutoModelForPreTraining, pipeline
import spacy
from google.colab import files

In [23]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = 0 if torch.cuda.is_available() else -1


##PDF and Text reader

In [15]:
#Get a file from user (will be  uploaded to drive ...)
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

Saving license.pdf to license.pdf


In [16]:
def extract_text_from_pdf(pdf_path):
    '''
    Opens a PDF file using PyMuPDF (fitz), iterates through each page,
    extracts the text content in plain text format, and returns the
    concatenated text as a single string.

    Args:
      pdf_path -> pdf file's route which in our case will be its name in colab enviroment
    '''
    doc = fitz.open(pdf_path)
    text = " "
    for page in doc:
        text += page.get_text("text") + "\n"
    return text

pdf_text = extract_text_from_pdf(file_name)

print(f"Extracted and processed text array from {file_name}\n{pdf_text}")

Extracted and processed text array from license.pdf
 Apache License, Version 2.0
Apache License
Version 2.0, January 2004
http://www.apache.org/licenses/
TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION
1. Definitions.
"License" shall mean the terms and conditions for use, reproduction,
and distribution as defined by Sections 1 through 9 of this document.
"Licensor" shall mean the copyright owner or entity authorized by
the copyright owner that is granting the License.
"Legal Entity" shall mean the union of the acting entity and all
other entities that control, are controlled by, or are under common
control with that entity. For the purposes of this definition,
"control" means (i) the power, direct or indirect, to cause the
direction or management of such entity, whether by contract or
otherwise, or (ii) ownership of fifty percent (50%) or more of the
outstanding shares, or (iii) beneficial ownership of such entity.
"You" (or "Your") shall mean an individual or Legal Entity

##Load the model, tockenizer, and pipeline from hugginface

In [5]:
tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model = AutoModelForPreTraining.from_pretrained("nlpaueb/legal-bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [6]:
#English pipeline optimized for CPU.
nlp = spacy.load("en_core_web_sm")

In [7]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", tokenizer="facebook/bart-large-cnn")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [23]:
def summarize_long_text(text, max_chunk_length=1024):
    """
    Splits long text into chunks and summarizes each chunk.
    """

    summaries = []
    # Split the text into chunks of a maximum length (e.g., 1024 tokens for BART)
    for i in range(0, len(text), max_chunk_length):
        chunk = text[i:i + max_chunk_length]
        # Generate summary for each chunk
        generated_result = summarizer(
            chunk,
            max_length=200,
            min_length=100,
            do_sample=False
        )

        summaries.append(generated_result[0]["summary_text"])

    # Join the summaries of chunks
    return " ".join(summaries)

# Generate the summary using the function for long texts
summary = summarize_long_text(pdf_text)
print(f"Legal Document's Summary:\n{summary}")
print(f"Length of the Summary: {len(summary)}")
print(f"Length of the Original Text: {len(pdf_text)}")

Your max_length is set to 200, but your input_length is only 128. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=64)


Legal Document's Summary:
Apache License, Version 2.0, January 2004. Terms and conditions for use, reproduction, and distribution as defined by Sections 1 through 9 of this document. Use of this License is subject to the terms and conditions of the Apache License, and may be changed by the Apache Software Foundation or a third party. The License is intended to provide a framework for the development of software. It is intended that no part of the License should be used for anything other than its intended purpose. The Apache License is a non-exclusive, non-transferable, and non-circumventable license. "Work" shall mean the work of authorship, whether in Source or Object form, made available under the License. "Derivative Works" shall include any work that is based on (or derived from) the Work and for which the editors, annotations, elaborations, or other modifications are intended. For the purposes of this License, Derivative works shall not include works that remain after the work ha

In [22]:
doc = nlp(pdf_text)
entites = {ent.label_: [] for ent in doc.ents}
for ent in doc.ents:
   entites[ent.label_].append(ent.text)

for key, values in entites.items():
    if values:
        print(f"\n<{key}>\n: {', '.join(values)}")


<ORG>
: Apache License, TERMS, Sections 1 through, License, License, License, The Apache Software Foundation, Legal Entity, License, License, Apache License, The Apache Software Foundation, License, NOTICE, the Derivative Works, NOTICE, the Derivative Works, the Derivative Works, NOTICE, License, NOTICE, License, License, License, Contributions, License, NOTICE, Contributions, TITLE, The Apache Software Foundation, License, Accepting Warranty or, License, TERMS, CONDITIONS, the Apache License, the Apache License, the Apache License, License, License, License, License, The Apache Software Foundation, The Apache Software Foundation

<CARDINAL>
: 2.0, 1, 9, 2, 3, 4, 2.0, at least one, 5, 6, 7, 2.0, 8, 9, 2.0, 2.0

<DATE>
: 2.0, January 2004, 2001-2007, Licensor, Licensor, 2001-2007, Licensor, Licensor, Licensor, 2001-2007, 2001-2007, 2001-2007

<PERSON>
: DISTRIBUTION, Appendix, Licensor, Copyright License, License, Derivative Works, License, Apache License

<WORK_OF_ART>
: Licensor, Leg

In [24]:
clauses = {
        "Payment Terms": [],
        "Confidentiality": [],
        "Termination": [],
        "Governing Law": []
}
for line in pdf_text.split("\n"):
    if "pay" in line.lower() or "compensation" in line.lower():
        clauses["Payment Terms"].append(line)
    elif "confidential" in line.lower() or "disclose" in line.lower():
        clauses["Confidentiality"].append(line)
    elif "terminate" in line.lower():
        clauses["Termination"].append(line)
    elif "law" in line.lower() or "jurisdiction" in line.lower():
         clauses["Governing Law"].append(line)

risk_keywords = ["breach", "liability", "penalty", "damages", "termination without cause"]
risks = []
for category, clause_list in clauses.items():
    for clause in clause_list:
       if any(word in clause.lower() for word in risk_keywords):
          risks.append(f"Possible Risk in {category}: {clause}")

print("\n Extracted Clauses")
for key, values in clauses.items():
    if values:
        print(f"\n {key}:")
        for clause in values:
            print(f"-{clause}")

print("\n Risk Analysis")
if risks:
    for risk in risks:
        print(risk)
else:
    print("No high risk clauses detected")


 Extracted Clauses

 Termination:
-granted to You under this License for that Work shall terminate

 Governing Law:
-cross-claim or counterclaim in a lawsuit) alleging that the Work
-7. Disclaimer of Warranty. Unless required by applicable law or
-unless required by applicable law (such as deliberate and grossly
-Unless required by applicable law or agreed to in writing, software

 Risk Analysis
No high risk clauses detected
